> Os exemplos desta página usam dados sintéticos gerados pelo projeto. Não são observações ASTER nem permitem interpretar valores como reflectância, temperatura ou classes reais. Veja [os dados de exemplo](../../../../data/README.md).



# Capítulo 07 — Classificação Supervisionada (Parte 2)

> 📖 **Referência:** Canty (2019), Cap. 7  
> Notas: [`aulas/redes_e_avaliacao.md`](../aulas/redes_e_avaliacao.md)  
> Avaliação no livro: `legado/canty/auxil/scripts/crossvalidate.py`, `mcnemar.py`

## 1. Intuição

Além do ML gaussiano: **SVM** (margem + kernel), **florestas/boosting** e **redes**. A escolha depende de tamanho amostral, linearidade das fronteiras e custo computacional.

Avaliação: acurácia, kappa, matriz de confusão; McNemar para comparar dois mapas na mesma amostra de teste. Cuidado com validação que ignora autocorrelação espacial.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import rasterio
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, cohen_kappa_score, classification_report

REPO = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "study_utils").is_dir())
EXEMPLO = REPO / 'data/exemplos/exemplo_sintetico.tif'
OUT = REPO / "conteudo/classificacao/redes-e-avaliacao/figuras"; OUT.mkdir(exist_ok=True)

In [ ]:
with rasterio.open(EXEMPLO) as ds:
    img = ds.read().astype(np.float64)
ndvi = (img[2]-img[1])/(img[2]+img[1]+1e-6)
y = np.zeros(ndvi.shape, dtype=int)
y[ndvi>=0.35]=2; y[(ndvi>=0.1)&(ndvi<0.35)]=1
X = img.reshape(img.shape[0], -1).T
idx = np.random.default_rng(1).choice(X.shape[0], min(10000, X.shape[0]), replace=False)
Xs, ys = X[idx], y.ravel()[idx]
Xtr, Xte, ytr, yte = train_test_split(Xs, ys, test_size=0.3, stratify=ys, random_state=0)
sc = StandardScaler().fit(Xtr)

clfs = {
    'NB': GaussianNB(),
    'RF': RandomForestClassifier(n_estimators=60, max_depth=10, random_state=0, n_jobs=-1),
}
for name, clf in clfs.items():
    clf.fit(sc.transform(Xtr), ytr)
    yp = clf.predict(sc.transform(Xte))
    print(name, 'acc', accuracy_score(yte, yp), 'kappa', cohen_kappa_score(yte, yp))
    print(classification_report(yte, yp, digits=3))

In [ ]:
pred = clfs['RF'].predict(sc.transform(X)).reshape(ndvi.shape)
fig, ax = plt.subplots(1,2,figsize=(9,4))
ax[0].imshow(ndvi, cmap='RdYlGn'); ax[0].set_title('NDVI-like')
ax[1].imshow(pred, cmap='tab10'); ax[1].set_title('RF')
for a in ax: a.axis('off')
fig.savefig(OUT/'nb07_rf_map.png', dpi=120, bbox_inches='tight')
plt.show()

## Próximos passos

Rode `exercicios/ex07_1.py` para NB vs RF (+ SVM quando couber) e o esboço de McNemar. Para redes do livro, veja o Apêndice B e `classify.py`.